# Step 1 = Import Libraries

In [51]:
import pandas as pd

from loader import load_tables

# Step 2 = Load Tables

In [74]:
tables = load_tables()

# Step 3 = Verify Tables

In [72]:
print(len(tables))

21


# Step 4 = Check name of tables

In [3]:
tables.keys()

dict_keys(['Billing_Invoice_Fact', 'Claim_Processing_Fact', 'Cost_Center_Master', 'Date_Dimension', 'Department_Master', 'Employee_Master', 'Finance_Fact', 'GL_Account_Master', 'Insurance_Master', 'Patient_Encounter_Fact', 'Patient_Master', 'Payer_Master', 'Payment_AR_Fact', 'Procedure_Master', 'Productivity_Performance_Fact', 'Provider_Master', 'Reference_Master', 'Region_Master', 'Transaction_Identity_Fact', 'Validation_Fact', 'Vendor_Master'])

# Step 5 = Check top 5 row of any table

In [4]:
Patient_Master = tables["Patient_Master"]

Patient_Master.head()

,Patient_ID,MRN,Insurance_ID,First_Name,Last_Name,Full_Name,DOB,Age,Gender,Blood_Group,...,High_Risk_Patient_Flag,Critical_Case_Flag,Chronic_Disease_Flag,VIP_Patient_Flag,Duplicate_Patient_Flag,Contact_Validation_Status,Demographic_Validation_Status,Patient_Status,Created_Date,Updated_Date
0,PAT000001,MRN0000001,INS-058,Rahul,Verma,Rahul Verma,1947-03-31,78,Male,A+,...,Yes,No,No,No,No,Validated,Validated,Active,2023-05-17,2023-12-17
1,PAT000002,MRN0000002,INS-041,Vihaan,Joshi,Vihaan Joshi,1997-11-13,28,Female,B+,...,No,No,Yes,No,No,Validated,Validated,Active,2022-01-14,2022-07-26
2,PAT000003,MRN0000003,INS-148,Vivaan,Kapoor,Vivaan Kapoor,1971-04-21,54,Male,A+,...,No,No,No,No,No,Validated,Validated,Active,2024-01-11,2024-05-26
3,PAT000004,MRN0000004,INS-063,Priya,Verma,Priya Verma,1999-08-26,26,Male,O+,...,Yes,No,No,No,No,Validated,Validated,Active,2024-12-29,2025-06-08
4,PAT000005,MRN0000005,INS-068,Vihaan,Joshi,Vihaan Joshi,1980-06-30,45,Male,A-,...,No,No,No,No,No,Validated,Validated,Active,2022-10-20,2023-01-09


# Step 6 = Manualy audit any table

In [44]:
df = tables["Patient_Master"]

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 2500
Columns: 33


# Step 7 = Create Function for automatic audit table

In [47]:
def audit_table(df, table_name):

    # -------------------------------
    # Basic Information
    # -------------------------------

    rows, columns = df.shape

    # -------------------------------
    # Missing Values
    # -------------------------------

    missing_values = df.isnull().sum().sum()

    missing_percent = round(
        (missing_values / (rows * columns)) * 100, 2
    ) if rows * columns else 0

    # -------------------------------
    # Duplicate Records
    # -------------------------------

    duplicate_rows = df.duplicated().sum()

    duplicate_percent = round(
        (duplicate_rows / rows) * 100, 2
    ) if rows else 0

    # -------------------------------
    # Data Types
    # -------------------------------

    numeric_columns = df.select_dtypes(include="number").shape[1]

    object_columns = df.select_dtypes(include="object").shape[1]

    datetime_columns = df.select_dtypes(
        include=["datetime64[ns]", "datetime64"]
    ).shape[1]

    # -------------------------------
    # Memory Usage
    # -------------------------------

    memory_usage_mb = round(
        df.memory_usage(deep=True).sum() / (1024 ** 2),
        2
    )

    # -------------------------------
    # Unique Values
    # -------------------------------

    unique_values = df.nunique().sum()

    # -------------------------------
    # Business Keys
    # -------------------------------

    business_keys = [
        col for col in df.columns
        if col.endswith("_ID")
    ]

    # -------------------------------
    # Quality Score
    # -------------------------------

    quality_score = round(
        100 - (missing_percent + duplicate_percent),
        2
    )

    quality_score = max(0, quality_score)

    # -------------------------------
    # Status
    # -------------------------------

    if quality_score >= 95:
        status = "Excellent"

    elif quality_score >= 85:
        status = "Good"

    elif quality_score >= 70:
        status = "Needs Review"

    else:
        status = "Critical"

    # -------------------------------
    # Return Summary
    # -------------------------------

    return {

        "Table Name": table_name,

        "Rows": rows,

        "Columns": columns,

        "Missing Values": missing_values,

        "Missing %": missing_percent,

        "Duplicate Rows": duplicate_rows,

        "Duplicate %": duplicate_percent,

        "Numeric Columns": numeric_columns,

        "Object Columns": object_columns,

        "Datetime Columns": datetime_columns,

        "Unique Values": unique_values,

        "Business Keys": ", ".join(business_keys),

        "Memory (MB)": memory_usage_mb,

        "Quality Score": quality_score,

        "Status": status

    }

# Step 8 = Create "For" Loop for Audit all tables automaticaly

In [49]:
audit_results = []

for table_name, df in tables.items():

    audit_results.append(
        audit_table(df, table_name)
    )

audit_summary = pd.DataFrame(audit_results)

audit_summary

,Table Name,Rows,Columns,Missing Values,Missing %,Duplicate Rows,Duplicate %,Numeric Columns,Object Columns,Datetime Columns,Unique Values,Business Keys,Memory (MB),Quality Score,Status
0,Billing_Invoice_Fact,520,10,0,0.0,0,0.0,5,5,0,2758,"Transaction_ID, Invoice_ID",0.17,100.0,Excellent
1,Claim_Processing_Fact,520,10,0,0.0,0,0.0,1,9,0,1070,"Transaction_ID, Claim_ID, Electronic_Payer_ID",0.30,100.0,Excellent
2,Cost_Center_Master,12,8,0,0.0,0,0.0,0,8,0,57,"Cost_Center_ID, Department_ID, Region_ID",0.01,100.0,Excellent
3,Date_Dimension,365,15,0,0.0,0,0.0,7,8,0,888,,0.17,100.0,Excellent
4,Department_Master,12,8,0,0.0,0,0.0,0,8,0,62,"Department_ID, Cost_Center_ID",0.01,100.0,Excellent
5,Employee_Master,300,17,0,0.0,0,0.0,5,12,0,2201,"Employee_ID, Department_ID, Region_ID",0.21,100.0,Excellent
6,Finance_Fact,520,10,0,0.0,0,0.0,6,4,0,2703,"Transaction_ID, Invoice_ID",0.14,100.0,Excellent
7,GL_Account_Master,10,8,0,0.0,0,0.0,1,7,0,48,"GL_Account_ID, Department_ID",0.00,100.0,Excellent
8,Insurance_Master,150,17,0,0.0,0,0.0,5,12,0,1045,Insurance_ID,0.11,100.0,Excellent
9,Patient_Encounter_Fact,520,9,0,0.0,0,0.0,1,8,0,1349,"Transaction_ID, Patient_ID, Procedure_ID",0.24,100.0,Excellent


# Step 9 = Export Audit Report

In [93]:
audit_summary.to_csv(
    "Audit_Report.csv",
    index=False
)

print("Audit_Report.csv saved successfully.")

Audit_Report.csv saved successfully.
